# Cubic Splines and Boundary Conditions

Piecewise polynomial interpolation.

A single high-degree polynomial through many points can diverge, as Runge's phenomenon shows. Splines instead use low-degree pieces joined with prescribed smoothness. Here we build piecewise-linear and cubic splines, look at the tridiagonal system that defines the cubic, and see what the boundary condition does.

In [ ]:
%pip install ipywidgets

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## The tridiagonal system

Let $M_i = s''(x_i)$ denote the second derivatives at the nodes and $h_i = x_{i+1}-x_i$. Continuity of the first derivative at each interior node gives, for $i = 1,\dots,n-1$,

$$ h_{i-1} M_{i-1} + 2(h_{i-1}+h_i) M_i + h_i M_{i+1} = 6\left(\frac{y_{i+1}-y_i}{h_i} - \frac{y_i-y_{i-1}}{h_{i-1}}\right). $$

This is a tridiagonal system in the $M_i$, solvable in $\mathcal{O}(n)$. It supplies $n-1$ equations for $n+1$ unknowns; the two remaining equations are the boundary conditions, either natural ($M_0=M_n=0$) or clamped (prescribed end slopes).

In [ ]:
# ---------------------------------------------------------------------------
# Piecewise-linear interpolation
# ---------------------------------------------------------------------------
# The simplest spline connects the dots with straight segments. It is continuous
# but has kinks, since its first derivative jumps at the nodes. Cubic splines below
# remove those kinks by matching first and second derivatives too.

def piecewise_linear(x_nodes, y_nodes, x):
    """Evaluate the connect-the-dots interpolant, which is what np.interp does."""
    return np.interp(x, x_nodes, y_nodes)

In [ ]:
# ---------------------------------------------------------------------------
# Cubic spline: solve for the second derivatives via a tridiagonal system
# ---------------------------------------------------------------------------
# A cubic spline is a cubic on each interval [x_i, x_{i+1}], stitched so that the
# function, its first derivative, and its second derivative are continuous at the
# interior nodes. The unknowns are the second derivatives M_i = s''(x_i). Interior
# continuity gives a tridiagonal linear system for the M_i; the two leftover
# degrees of freedom are fixed by one of these boundary conditions.
#
#   "natural"  : M_0 = M_n = 0            (zero curvature at the ends)
#   "clamped"  : prescribe s'(x_0), s'(x_n)   (here we clamp to the data's slope)

def cubic_spline_moments(x, y, bc="natural", slopes=None):
    """Return the second-derivative moments M_i defining the cubic spline.

    Parameters
    ----------
    x, y   : arrays (n+1,)  sorted nodes and values
    bc     : "natural" or "clamped"
    slopes : (s'(x_0), s'(x_n)) for the clamped case. Defaults to the secant
             slopes of the first and last panel, which clamps to the data.

    Returns
    -------
    M : array (n+1,)  second derivatives s''(x_i)
    """
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x) - 1                        # number of intervals
    h = np.diff(x)                        # interval widths h_i = x_{i+1}-x_i

    # Assemble the (n+1)x(n+1) tridiagonal system  A M = d.
    A = np.zeros((n + 1, n + 1))
    d = np.zeros(n + 1)
    # Interior equations, from continuity of the first derivative.
    for i in range(1, n):
        A[i, i - 1] = h[i - 1]
        A[i, i]     = 2 * (h[i - 1] + h[i])
        A[i, i + 1] = h[i]
        d[i] = 6 * ((y[i + 1] - y[i]) / h[i] - (y[i] - y[i - 1]) / h[i - 1])

    if bc == "natural":
        A[0, 0] = 1.0; d[0] = 0.0         # M_0 = 0
        A[n, n] = 1.0; d[n] = 0.0         # M_n = 0
    elif bc == "clamped":
        if slopes is None:                # default, clamp to the end secants
            slopes = ((y[1] - y[0]) / h[0], (y[n] - y[n - 1]) / h[n - 1])
        fp0, fpn = slopes
        A[0, 0] = 2 * h[0]; A[0, 1] = h[0]
        d[0] = 6 * ((y[1] - y[0]) / h[0] - fp0)
        A[n, n - 1] = h[n - 1]; A[n, n] = 2 * h[n - 1]
        d[n] = 6 * (fpn - (y[n] - y[n - 1]) / h[n - 1])
    else:
        raise ValueError("bc must be 'natural' or 'clamped'")

    return np.linalg.solve(A, d)


def cubic_spline_eval(x, y, M, xq):
    """Evaluate the cubic spline (given moments M) at query points xq.

    On interval [x_i, x_{i+1}] the spline is the standard moment formula
    combining M_i, M_{i+1} and the endpoint values.
    """
    x = np.asarray(x, float); y = np.asarray(y, float)
    xq = np.asarray(xq, float)
    h = np.diff(x)
    # For each query point find its interval index i (clip to a valid panel).
    idx = np.clip(np.searchsorted(x, xq) - 1, 0, len(x) - 2)
    out = np.empty_like(xq)
    for k, xk in enumerate(xq):
        i = idx[k]
        a = x[i + 1] - xk
        b = xk - x[i]
        hi = h[i]
        out[k] = (M[i] * a**3 + M[i + 1] * b**3) / (6 * hi) \
                 + (y[i] / hi - M[i] * hi / 6) * a \
                 + (y[i + 1] / hi - M[i + 1] * hi / 6) * b
    return out

In [ ]:
# ---------------------------------------------------------------------------
# Compare linear vs cubic, natural vs clamped
# ---------------------------------------------------------------------------
def f_demo(x):
    """Target function chosen so the boundary condition affects the ends."""
    return np.cos(x) * np.exp(-0.15 * x)

def show_spline(n_nodes=7, spline="cubic", bc="natural", a=0.0, b=10.0):
    """Sample f at n_nodes points and plot the chosen interpolant against f."""
    x = np.linspace(a, b, n_nodes)
    y = f_demo(x)
    xx = np.linspace(a, b, 600)

    plt.figure()
    plt.plot(xx, f_demo(xx), "k-", lw=1.5, alpha=0.6, label="f(x)")
    if spline == "linear":
        plt.plot(xx, piecewise_linear(x, y, xx), "g-", lw=2,
                 label="piecewise linear")
    else:
        M = cubic_spline_moments(x, y, bc=bc)
        plt.plot(xx, cubic_spline_eval(x, y, M, xx), "r-", lw=2,
                 label=f"cubic spline ({bc})")
    plt.plot(x, y, "bo", ms=6, label="nodes")
    plt.legend(); plt.xlabel("x")
    plt.title(f"{spline} interpolation" + (f", {bc} BC" if spline == "cubic" else ""))
    plt.show()

show_spline(7, "cubic", "natural")

In [ ]:
# ---------------------------------------------------------------------------
# Interactive: node count, linear vs cubic, and the boundary condition
# ---------------------------------------------------------------------------
# With few nodes, switch bc between "natural" and "clamped" and watch how the
# curve bends differently near the two ends. The interior is barely affected.
interact(
    show_spline,
    n_nodes=IntSlider(min=3, max=15, step=1, value=7, description="# nodes"),
    spline=Dropdown(options=["cubic", "linear"], value="cubic", description="type"),
    bc=Dropdown(options=["natural", "clamped"], value="natural", description="cubic BC"),
    a=(-2.0, 2.0, 1.0), b=(6.0, 14.0, 1.0),
);

## What the boundary condition costs

The natural condition asserts $s''=0$ at the ends, which is usually false for the function being interpolated, so it damages the fit there. The question is how far inward the damage reaches. The cell below reports the largest error on the end panels, over everything except the end panels, and over the middle third, for the natural condition, for the clamped condition with the exact end slopes, and for the linear spline. Then it refines the mesh and reports the factor by which each falls at every doubling.

In [ ]:
def df_demo(x):
    """Exact derivative of f_demo, so the clamped condition can be exact."""
    return np.exp(-0.15 * x) * (-np.sin(x) - 0.15 * np.cos(x))

def error_split(m):
    """Max error on the end panels, outside them, and over the middle third."""
    x = np.linspace(0.0, 10.0, m)
    y = f_demo(x)
    fine = np.linspace(0.0, 10.0, 4001)
    left, right = fine <= x[1], fine >= x[-2]
    inner = ~(left | right)                       # everything but the end panels
    middle = (fine > 10.0 / 3) & (fine < 20.0 / 3)  # a fixed fraction of [0, 10]

    def regions(e):
        return (e[left].max(), e[right].max(), e[inner].max(), e[middle].max())

    out = {}
    for name in ["natural", "clamped"]:
        M = cubic_spline_moments(x, y, bc=name,
                                 slopes=(df_demo(x[0]), df_demo(x[-1])))
        out[name] = regions(np.abs(f_demo(fine) - cubic_spline_eval(x, y, M, fine)))
    out["linear"] = regions(np.abs(f_demo(fine) - piecewise_linear(x, y, fine)))
    return out

print("m = 9 nodes")
for name, (lo, hi, inner, mid) in error_split(9).items():
    print(f"  {name:8s} first panel {lo:.3e}   last panel {hi:.3e}   "
          f"outside the end panels {inner:.3e}   middle third {mid:.3e}")

nan = float("nan")
for name in ["natural", "clamped"]:
    print(f"\n{name} condition, error and the factor it falls by at each doubling")
    print(f"{'m':>5} {'end panels':>19} {'outside them':>19} {'middle third':>19}")
    prev = None
    for m in [9, 17, 33, 65, 129]:
        r = error_split(m)
        end, out, mid = max(r[name][0], r[name][1]), r[name][2], r[name][3]
        f1, f2, f3 = (prev[0] / end, prev[1] / out, prev[2] / mid) if prev else (nan, nan, nan)
        print(f"{m:5d} {end:12.2e} (x{f1:4.1f}) {out:12.2e} (x{f2:4.1f})"
              f" {mid:12.2e} (x{f3:4.1f})")
        prev = (end, out, mid)

**Reading the numbers.** The natural error falls by about $4$ at each doubling and the clamped error by about $16$, so the rates are $\mathcal{O}(h^2)$ and $\mathcal{O}(h^4)$. That holds not only on the end panels but on everything outside them, so the natural condition costs two orders globally, not just locally. Over the middle third, though, the two conditions agree to every printed digit and both run at $\mathcal{O}(h^4)$. The reconciliation is that the boundary error decays by a fixed factor per panel, so a region measured in panels away from the end stays polluted while a region measured as a fraction of the interval sits ever more panels away as the mesh refines. Since $|f''(0)| \approx 1$ against $|f''(10)| \approx 0.15$ for this $f$, the left end takes more of the damage, which the first two columns show. Clamping is worth it when the end derivatives are known, and when they are not, clamping to the end secant slope is a guess that can be worse than natural.

## Against a single polynomial

Splines were introduced as an answer to Runge's phenomenon, so here is the direct comparison, both interpolants through the same 21 equally spaced samples.

In [ ]:
def runge(x):
    """Runge's function 1/(1 + 25 x^2) on [-1, 1]."""
    return 1.0 / (1.0 + 25.0 * x**2)

def lagrange_eval(nodes, values, x):
    """Evaluate the interpolating polynomial in Lagrange form."""
    nodes = np.asarray(nodes, dtype=float); values = np.asarray(values, dtype=float)
    x = np.asarray(x, dtype=float)
    result = np.zeros_like(x)
    for i in range(len(nodes)):
        Li = np.ones_like(x)
        for j in range(len(nodes)):
            if j != i:
                Li *= (x - nodes[j]) / (nodes[i] - nodes[j])
        result += values[i] * Li
    return result

xr = np.linspace(-1.0, 1.0, 21)
yr = runge(xr)
fine = np.linspace(-1.0, 1.0, 2001)
poly = lagrange_eval(xr, yr, fine)
spl = cubic_spline_eval(xr, yr, cubic_spline_moments(xr, yr, bc="natural"), fine)

print(f"degree 20 polynomial   max error {np.abs(runge(fine) - poly).max():.3e}")
print(f"natural cubic spline   max error {np.abs(runge(fine) - spl).max():.3e}")

plt.figure()
plt.plot(fine, runge(fine), "k-", lw=1.5, label="f")
plt.plot(fine, poly, "r--", lw=1.5, label="degree 20 polynomial")
plt.plot(fine, spl, "b-", lw=1.5, label="natural cubic spline")
plt.plot(xr, yr, "ko", ms=4)
plt.ylim(-1.0, 2.0); plt.xlabel("x"); plt.legend()
plt.title("21 equally spaced nodes"); plt.show()

**Why the spline escapes.** Raising the node count raises the polynomial degree, and with it $\max_x|\omega(x)|$ over the whole interval, which is what drives the equispaced divergence. The spline keeps degree three no matter how many nodes there are, and each data value influences only the panels next to it, so refining the mesh shrinks the error instead of redistributing it into endpoint oscillations.

## Summary

- Cubic splines match value, first derivative, and second derivative at interior nodes, avoiding both kinks and Runge divergence.
- The moments $M_i$ solve a tridiagonal system.
- The boundary condition fixes the two remaining degrees of freedom. Natural gives $\mathcal{O}(h^2)$ accuracy, clamping with the exact end slopes gives $\mathcal{O}(h^4)$, and the gap persists outside the end panels even though it vanishes over the middle of the interval.
- The linear spline is continuous but has a discontinuous first derivative.

## Things to try

- With few nodes, switch `bc` between natural and clamped and watch the two ends bend differently while the middle stays put.
- Switch to the linear spline and look for the kinks at the nodes, which are the visible sign that $s'$ jumps there.
- Replace `f_demo` with a function whose second derivative is large at the endpoints and see what the natural condition costs there.